## ***Rview Machine Learning***
---

In this docuemnt we are going to focus on the review of some topics and examples about machine learning seen in class. The goal is create my own prediction model applying the topics studied previously. 

### **Supervised learnign**
### **1. The classification**
---


In the classification is important that our data has already cleaned. we can't have *NaN, null or corrupt* values. Also is important apply the ***permutation***, to guarantee the data ramdomness

### **The Gaussian Naive Bayes**
---

#### **What is this?**
---
The gaussian naive bayes is a variant of the naive bayes algorithm used for clasification, when your input features are continuous and you assume that they follow a normal distribution.                                           

#### **When to use the gaussian naive bayes**
---
We use the naussian naive bayes when we have:

- Small datasets
- Features roughly normally distributed
- Real-time predictions (very fast)
- Baseline classifier

In [1]:
#Import all our resources
import numpy as np
import pandas as pd

## Information about our dataset
---


| Column | Description |
| :--- | :--- |
| **job_title** | The job role or position (e.g., Data Analyst, AI Engineer) |
| **experience_years** | Number of years of professional experience |
| **education_level** | Highest level of education completed |
| **skills_count** | Number of technical or professional skills |
| **industry** | Industry sector where the job belongs |
| **company_size** | Size of the company (small, medium, large) |
| **location** | Job location or region |
| **remote_work** | Whether the job allows remote work |
| **certifications** | Number of professional certifications |
| **salary** | Annual salary of the employee |


### **We need to extract the X and Y(ground truth) columns**

In [7]:
df = pd.read_csv("jobSalaryDataset/job_salary_prediction_dataset.csv")

bins = [0, 75000, 100000, float('inf')]
labels = ['Low', 'Medium', 'High']

df['salary_range'] = pd.cut(df['salary'], bins=bins, labels=labels)
df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary,salary_range
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413,High
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764,Medium
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123,High
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123,High
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069,High


In [8]:
X_values = df.loc[:,["experience_years", "skills_count", "company_size", "certifications"]]
y_values = df.loc[:,"salary_range"]

print(X_values.company_size.value_counts())
X_values["company_size"] = X_values["company_size"].replace({"Startup": 1, "Small": 2, "Medium": 3, "Large": 4, "Enterprise": 5})

X_values.head()

company_size
Large         50254
Small         50235
Medium        50027
Enterprise    49875
Startup       49609
Name: count, dtype: int64


,experience_years,skills_count,company_size,certifications
0,10,2,3,2
1,5,17,2,0
2,18,4,3,1
3,19,13,3,0
4,15,7,4,0


In [9]:
y_values.head()

0      High
1    Medium
2      High
3      High
4      High
Name: salary_range, dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']

#### **We need create the estimator,  and the train predict**
---
in this case we are going to use the gaussian naive bayes estimator
and the model selection cross_val_score to obtain a good permutation and keep the randomness.

In [10]:
#import resources from sklearn
from sklearn.naive_bayes import GaussianNB
#2 selection models
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#test_validation => Percentage of the data for test, in this case is 0.3 i mean 30%
#random_state => Reproducible output across multiple function calls
#shuffle => We can use it to permutate the data before split, but in this application we are using random state and it's not necessary
X_train, X_test, Y_train, Y_test = train_test_split(X_values, y_values, test_size=0.3, random_state=21)

#We need to create our estimator
est = GaussianNB()

#We need to train our model
#to do this we use the fit function
est.fit(X_train,Y_train)

#Now we already to predict the future values, this function returns an array with the predicted values
preVal = est.predict(X_test)
print(preVal)

#Now we can select a metric to compare the results of the predict with the real values
print(accuracy_score(Y_test, preVal))


['High' 'High' 'High' ... 'High' 'High' 'High']
0.8955866666666666


Now we are goint to change the selection model for a cross validation

In [11]:
#before we have imported the cross_val_score
#This function returns an array where we cand found different valus of a metric
#our first parameter is the estimator = for us is the est with Gaussian naive bayes
#the second parameter is the data to fit, i mean X witout split.
#the third parameter is the data objetive, our ground truth y without split.

#We import the KFold to indicates how many iterations we want and if we want to shuffle the data before splitting.
from sklearn.model_selection import KFold
accuracys = cross_val_score(est, X_values, y_values, scoring='accuracy', cv=KFold(10, shuffle=True) )

print(accuracys)
print("Mean: ", np.mean(accuracys))
print("Standard desviation: ", np.std(accuracys))

[0.8964  0.89392 0.89272 0.89608 0.89528 0.89436 0.89356 0.89184 0.89676
 0.8956 ]
Mean:  0.894652
Standard desviation:  0.001558363243919721
